In [ ]:
import torch
cap=torch.cuda.get_device_capability(0) if torch.cuda.is_available() else None
name=torch.cuda.get_device_name(0) if cap else "none"
P100=bool(cap and cap[0]==6)
print("GPU:",name,"| sm",cap,"| p100 mode:",P100)

In [ ]:
import subprocess,sys
def pip(*a):
    r=subprocess.run([sys.executable,"-m","pip","install","--no-warn-script-location",*a],capture_output=True,text=True)
    print(r.stdout[-400:]); print("rc=",r.returncode, r.stderr[-400:] if r.returncode else "(ok)")
    return r.returncode
if P100: pip("torch==2.1.2","--index-url","https://download.pytorch.org/whl/cu121")
pip("-U","--no-deps","transformers")
pip("-U","peft","trl","datasets","torchao")
print("---ground truth---")
subprocess.run([sys.executable,"-m","pip","show","torch","transformers","trl","peft"],check=False)

In [ ]:
from transformers import AutoTokenizer,AutoModelForCausalLM
import torch
B="DreamFast/qwen3-4b-heretic"
tok=AutoTokenizer.from_pretrained(B)
if tok.pad_token is None: tok.pad_token=tok.eos_token
m=AutoModelForCausalLM.from_pretrained(B,torch_dtype=torch.float16,device_map="auto",attn_implementation=("eager" if P100 else "sdpa"))
m.config.use_cache=False
print("loaded",B)

In [ ]:
import os,glob,json
p=None
for cand in ["/kaggle/input/enilo-rehan-v1/train_v1.jsonl"]+glob.glob("/kaggle/input/**/train_v1.jsonl",recursive=True):
    if os.path.exists(cand): p=cand; break
assert p,"attach dataset enilo-rehan-v1 to kernel"
from datasets import Dataset
rows=[json.loads(l) for l in open(p)]
def norm(r):
    out=[]
    for msg in r["messages"]:
        if msg.get("content") is None and msg.get("tool_calls"):
            tc=msg["tool_calls"][0]["function"]
            msg={"role":"assistant","content":"<tool_call name=\""+tc["name"]+"\">"+tc["arguments"]+"</tool_call>"}
        out.append(msg)
    return out
ds=Dataset.from_list([{"messages":norm(r)} for r in rows])
print("lessons:",len(ds))

In [ ]:
from transformers import TrainingArguments
from trl import SFTConfig,SFTTrainer
from peft import LoraConfig
pc=LoraConfig(r=16,lora_alpha=32,target_modules="all-linear",task_type="CAUSAL_LM")
cfg=SFTConfig(output_dir="/kaggle/working/run",per_device_train_batch_size=1,gradient_accumulation_steps=8,
 learning_rate=1.5e-4,num_train_epochs=3,logging_steps=5,max_length=1536,eos_token="<|im_end|>",loss_type="nll",fp16=True,bf16=False,
 save_strategy="no",report_to=[])
trk=SFTTrainer(model=m,args=cfg,train_dataset=ds,processing_class=tok,peft_config=pc)
print("trainer ready")

In [ ]:
trk.train()
print("train done")

In [ ]:
out="/kaggle/working/enilo-rehan-v1"
mg=trk.model.merge_and_unload() if hasattr(trk.model,"merge_and_unload") else trk.model
mg.save_pretrained(out)
tok.save_pretrained(out)
import os
tot=0
for r,_,fs in os.walk(out):
    for f in fs:
        s=os.path.getsize(os.path.join(r,f)); tot+=s
        print(f"\t{s//1_000_000}MB\t{f}")
print(f"TOTAL {tot/1e9:.2f}GB")